# LeetCode 851 — Loud and Rich bằng Python

Link đề: https://leetcode.com/problems/loud-and-rich/ · Độ khó: **Medium** · Chủ đề: Graph (đồ thị), DFS, Topological Sort (sắp xếp tô-pô), Directed Acyclic Graph (đồ thị có hướng không chu trình), Array

Skeleton LeetCode:

```python
class Solution:
    def loudAndRich(self, richer: List[List[int]], quiet: List[int]) -> List[int]:
        ...
```

Ghi chú: các quan hệ "`a` giàu hơn `b`" tạo thành một **DAG** (đồ thị có hướng không chu trình). Với mỗi người `x`, ta cần người **yên lặng nhất** (giá trị `quiet` nhỏ nhất) trong **tất cả những người chắc chắn giàu bằng hoặc hơn `x`** (kể cả chính `x` và những người giàu hơn **gián tiếp** qua chuỗi quan hệ). Notebook đi từ duyệt riêng cho từng người, qua DFS có nhớ, sắp xếp tô-pô, bitset bao đóng bắc cầu, tới cách "xử lý người yên lặng nhất trước".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi người có một mức giàu (ẩn) và một mức yên lặng `quiet[i]` (**đôi một khác nhau**); `richer[i] = [a, b]` nghĩa là `a` giàu hơn `b`; với mỗi `x` tìm `y` yên lặng nhất trong tập những người **chắc chắn** giàu bằng hoặc hơn `x`.
2. Nắm các công cụ Python cần thiết: danh sách kề (`defaultdict(list)`), DFS đệ quy có nhớ, hàng đợi `deque` và bậc vào (in-degree) cho sắp xếp tô-pô, **số nguyên làm bitset** (`1 << i`, `|`, duyệt bit), `sorted(..., key=...)`.
3. Hiểu "chắc chắn giàu hơn" = **đến được theo các cạnh `giàu -> nghèo`** (bắc cầu); và câu trả lời của `x` là `min` của `chính x` và các câu trả lời của **những người giàu hơn trực tiếp** (quy hoạch động trên DAG).
4. Đi qua 5 cách: duyệt riêng từng người, DFS có nhớ, sắp xếp tô-pô, bao đóng bắc cầu bằng bitset, xử lý người từ yên lặng nhất tới ồn nhất rồi lan sang những người nghèo hơn; đối chiếu bằng oracle.
5. Tránh các lỗi kinh điển: đặt ngược chiều cạnh, quên chính `x` trong tập ứng viên, chỉ xét người giàu hơn trực tiếp (không bắc cầu), so sánh theo chỉ số thay vì `quiet`, và không ghi nhớ khiến DFS chạy mũ.

## 2. Đề bài & ví dụ

Có một nhóm `n` người đánh số `0..n-1`, mỗi người có số tiền **khác nhau** và độ yên lặng **khác nhau**.

Cho mảng `richer`: `richer[i] = [a_i, b_i]` nghĩa là `a_i` có nhiều tiền hơn `b_i`; và mảng `quiet`: `quiet[i]` là độ yên lặng của người thứ `i` (**giá trị nhỏ hơn nghĩa là yên lặng hơn**). Mọi dữ liệu `richer` đều **nhất quán** (không có chu trình: không có `x` giàu hơn `y` mà `y` lại giàu hơn `x`).

Trả về mảng `answer` với `answer[x] = y` nếu `y` là người **yên lặng nhất** (giá trị `quiet[y]` nhỏ nhất) trong **tất cả những người chắc chắn có tiền bằng hoặc nhiều hơn `x`**.

**Ví dụ 1**

- Input: `richer = [[1,0],[2,1],[3,1],[3,7],[4,3],[5,3],[6,3]]`, `quiet = [3,2,5,4,6,1,7,0]`
- Output: `[5,5,2,5,4,5,6,7]`
- Giải thích:
  - `answer[0] = 5`: người 5 giàu hơn 3, người 3 giàu hơn 1, người 1 giàu hơn 0. Người duy nhất yên lặng hơn 5 là người 7 nhưng không rõ 7 có giàu hơn 0 hay không.
  - `answer[7] = 7`: trong những người chắc chắn giàu bằng hoặc hơn 7 (là 3, 4, 5, 6, 7), người yên lặng nhất là 7.

**Ví dụ 2**

- Input: `richer = []`, `quiet = [0]`
- Output: `[0]`

**Ràng buộc:**

- `n == quiet.length`, `1 <= n <= 500`
- `0 <= quiet[i] < n`; mọi giá trị `quiet` **khác nhau**.
- `0 <= richer.length <= n * (n - 1) / 2`
- `0 <= a_i, b_i < n`, `a_i != b_i`; mọi cặp trong `richer` khác nhau; dữ liệu nhất quán.

In [1]:
from typing import List
from collections import defaultdict, deque
import random, time, sys

sys.setrecursionlimit(10000)

examples = [
    ([[1, 0], [2, 1], [3, 1], [3, 7], [4, 3], [5, 3], [6, 3]], [3, 2, 5, 4, 6, 1, 7, 0], [5, 5, 2, 5, 4, 5, 6, 7]),
    ([], [0], [0]),
]
for richer, quiet, expected in examples:
    print(f"richer={richer} quiet={quiet} kỳ vọng={expected}")

richer=[[1, 0], [2, 1], [3, 1], [3, 7], [4, 3], [5, 3], [6, 3]] quiet=[3, 2, 5, 4, 6, 1, 7, 0] kỳ vọng=[5, 5, 2, 5, 4, 5, 6, 7]
richer=[] quiet=[0] kỳ vọng=[0]


## 3. Kiến thức nền

### 3.1. Đồ thị có hướng và danh sách kề

Mỗi cặp `[a, b]` là một **cạnh có hướng**. Ta cần hỏi "ai giàu hơn `x`?", nên ta lưu cạnh **theo chiều từ người nghèo hơn tới người giàu hơn**: `g[b].append(a)` (danh sách những người **giàu hơn trực tiếp** `b`). `defaultdict(list)` tạo list rỗng cho khóa mới.

In [2]:
richer, quiet, _ = examples[0]
g = defaultdict(list)
for a, b in richer:
    g[b].append(a)
for x in range(len(quiet)):
    print(f"người {x}: giàu hơn trực tiếp là {g[x]}")

người 0: giàu hơn trực tiếp là [1]
người 1: giàu hơn trực tiếp là [2, 3]
người 2: giàu hơn trực tiếp là []
người 3: giàu hơn trực tiếp là [4, 5, 6]
người 4: giàu hơn trực tiếp là []
người 5: giàu hơn trực tiếp là []
người 6: giàu hơn trực tiếp là []
người 7: giàu hơn trực tiếp là [3]


### 3.2. Bắc cầu (transitive) và DAG

Nếu `a` giàu hơn `b` và `b` giàu hơn `c` thì `a` **chắc chắn** giàu hơn `c` dù cặp `(a, c)` không được cho. Tập những người chắc chắn giàu hơn `x` là mọi đỉnh **đến được** từ `x` khi đi theo các cạnh `g` (poorer -> richer). Vì dữ liệu nhất quán nên đồ thị **không có chu trình** (DAG): DFS không bao giờ quay lại chính nó.

In [3]:
def richer_than(x):
    """Mọi người chắc chắn giàu hơn x (bắc cầu), duyệt DFS theo g."""
    seen, stack = set(), [x]
    while stack:
        u = stack.pop()
        for v in g[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return sorted(seen)

for x in (0, 1, 7):
    print(f"giàu hơn {x} (chắc chắn):", richer_than(x))

giàu hơn 0 (chắc chắn): [1, 2, 3, 4, 5, 6]
giàu hơn 1 (chắc chắn): [2, 3, 4, 5, 6]
giàu hơn 7 (chắc chắn): [3, 4, 5, 6]


### 3.3. DFS đệ quy và ghi nhớ (memoization)

DFS đệ quy gọi lại chính nó trên các láng giềng. Nếu mỗi đỉnh được tính **nhiều lần** (vì nhiều đường dẫn tới nó), ta lưu kết quả vào mảng `ans` và **bỏ qua khi đã có** (`ans[i] != -1`). Khi đó mỗi đỉnh tính đúng một lần và tổng thời gian `O(n + E)`.

In [4]:
calls = 0
@__import__("functools").cache
def fib(k):
    global calls
    calls += 1
    return k if k < 2 else fib(k - 1) + fib(k - 2)
print(fib(25), "- số lần thân hàm chạy thực sự:", calls, "(không nhớ sẽ là ~250.000)")

75025 - số lần thân hàm chạy thực sự: 26 (không nhớ sẽ là ~250.000)


### 3.4. Sắp xếp tô-pô và bậc vào (Kahn)

**Sắp xếp tô-pô** là thứ tự các đỉnh sao cho mọi cạnh `u -> v` có `u` đứng trước `v`. Thuật toán Kahn: tính **bậc vào** (số cạnh trỏ vào) của mỗi đỉnh; đưa các đỉnh bậc vào 0 vào hàng đợi; mỗi lần lấy `u` ra, giảm bậc vào của các `v` kề `u`, đỉnh nào về 0 thì đưa vào hàng đợi. Với cạnh `giàu -> nghèo`, thứ tự tô-pô là **từ giàu nhất tới nghèo nhất**.

In [5]:
def topo_rich_to_poor(n, richer):
    out = defaultdict(list)                 # giàu -> nghèo
    indeg = [0] * n
    for a, b in richer:
        out[a].append(b)
        indeg[b] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in out[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order

print(topo_rich_to_poor(8, examples[0][0]))

[2, 4, 5, 6, 3, 1, 7, 0]


### 3.5. Số nguyên làm bitset

Với `n <= 500`, một **tập các người** có thể lưu bằng **một số nguyên** `mask` (bit `i` bằng 1 nghĩa là người `i` thuộc tập). Python có số nguyên lớn tùy ý nên `1 << 499` hoàn toàn hợp lệ. `a | b` là hợp tập, `mask >> i & 1` kiểm tra thuộc tập, `mask & -mask` lấy bit thấp nhất.

In [6]:
mask = 0
for p in (1, 3, 7):
    mask |= 1 << p
print(bin(mask), [i for i in range(10) if mask >> i & 1], "số phần tử:", mask.bit_count())
a, b = (1 << 2) | (1 << 5), (1 << 5) | (1 << 9)
print("hợp:", [i for i in range(12) if (a | b) >> i & 1])

0b10001010 [1, 3, 7] số phần tử: 3
hợp: [2, 5, 9]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét `x = 0` ở ví dụ 1. Ai chắc chắn giàu bằng hoặc hơn 0? Theo chuỗi `1 -> 0`, `2 -> 1`, `3 -> 1`, `4 -> 3`, `5 -> 3`, `6 -> 3`, `3 -> 7`: giàu hơn 0 trực tiếp là `1`; giàu hơn 1 là `2` và `3`; giàu hơn 3 là `4, 5, 6`. Vậy tập ứng viên của 0 là `{0, 1, 2, 3, 4, 5, 6}` (người 7 không có trong đó). Độ yên lặng: `quiet = [3, 2, 5, 4, 6, 1, 7, 0]` -> trong tập, người 5 có `quiet = 1` nhỏ nhất. `answer[0] = 5` ✓.

Quan sát then chốt: ứng viên của `x` = `{x}` ∪ ứng viên của **mọi người giàu hơn `x` trực tiếp**. Do đó

`answer[x] = người có quiet nhỏ nhất trong {x} ∪ {answer[y] : y giàu hơn x trực tiếp}`

vì `answer[y]` đã là người yên lặng nhất trong ứng viên của `y`, ta chỉ cần so các đại diện.

In [7]:
richer, quiet, expected = examples[0]
n = len(quiet)
direct = defaultdict(list)
for a, b in richer:
    direct[b].append(a)

def candidates(x):
    seen, stack = {x}, [x]
    while stack:
        u = stack.pop()
        for v in direct[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return seen

for x in range(n):
    cs = candidates(x)
    best = min(cs, key=lambda p: quiet[p])
    print(f"x={x}: ứng viên {sorted(cs)} -> yên lặng nhất {best} (quiet={quiet[best]})")
    assert best == expected[x]

x=0: ứng viên [0, 1, 2, 3, 4, 5, 6] -> yên lặng nhất 5 (quiet=1)
x=1: ứng viên [1, 2, 3, 4, 5, 6] -> yên lặng nhất 5 (quiet=1)
x=2: ứng viên [2] -> yên lặng nhất 2 (quiet=5)
x=3: ứng viên [3, 4, 5, 6] -> yên lặng nhất 5 (quiet=1)
x=4: ứng viên [4] -> yên lặng nhất 4 (quiet=6)
x=5: ứng viên [5] -> yên lặng nhất 5 (quiet=1)
x=6: ứng viên [6] -> yên lặng nhất 6 (quiet=7)
x=7: ứng viên [3, 4, 5, 6, 7] -> yên lặng nhất 7 (quiet=0)


## 5. Cách 1 — Brute Force: duyệt riêng cho từng người

**Ý tưởng.** Với mỗi `x`, chạy một lần duyệt (DFS/BFS) trên danh sách "giàu hơn trực tiếp" để thu thập mọi người chắc chắn giàu hơn (bắc cầu), cộng chính `x`, rồi lấy người có `quiet` nhỏ nhất. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [8]:
def loud_brute(richer: List[List[int]], quiet: List[int]) -> List[int]:
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    ans = []
    for x in range(n):
        seen, stack = {x}, [x]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
        ans.append(min(seen, key=lambda p: quiet[p]))
    return ans

for richer, quiet, expected in examples:
    got = loud_brute(richer, quiet)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 5, 2, 5, 4, 5, 6, 7] OK
[0] OK


**Độ phức tạp và nút thắt.** Mỗi `x` một lần duyệt `O(n + E)`: tổng `O(n (n + E))`, với `n = 500` và `E` tới `~125.000` là cỡ `6 * 10^7`: chậm. Nút thắt: các lần duyệt **đi lại cùng những đường** (nếu `x` và `x'` cùng dưới một người giàu `y`, ta duyệt lại toàn bộ phần phía trên `y`). Công thức đệ quy ở mục 4 cho phép dùng lại kết quả.

In [9]:
def random_dag(rng, n, p):
    """DAG nhất quán: chọn một thứ tự giàu-nghèo ngẫu nhiên và các cạnh giàu -> nghèo với xác suất p."""
    order = list(range(n))
    rng.shuffle(order)                         # order[0] giàu nhất
    richer = [[order[i], order[j]] for i in range(n) for j in range(i + 1, n) if rng.random() < p]
    quiet = list(range(n))
    rng.shuffle(quiet)
    return richer, quiet

rng = random.Random(1)
for n, p in [(100, 0.3), (200, 0.3), (300, 0.3), (500, 0.1)]:
    richer, quiet = random_dag(rng, n, p)
    t0 = time.perf_counter()
    loud_brute(richer, quiet)
    print(f"n={n:3} số cạnh={len(richer):6}: vét cạn mất {1000 * (time.perf_counter() - t0):9.1f} ms")

n=100 số cạnh=  1477: vét cạn mất       3.0 ms
n=200 số cạnh=  6033: vét cạn mất      16.4 ms
n=300 số cạnh= 13369: vét cạn mất      52.6 ms


n=500 số cạnh= 12285: vét cạn mất     107.1 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Dùng lại kết quả của người giàu hơn trực tiếp.** Theo mục 4, `answer[x]` chỉ phụ thuộc `x` và `answer[y]` của các người giàu hơn **trực tiếp**. Đó là một **quy hoạch động trên DAG**: tính theo thứ tự mà mọi người giàu hơn đã có kết quả (thứ tự tô-pô giàu -> nghèo), hoặc dùng DFS có nhớ.

Ngoài ra còn hai cách nhìn khác:

- **Bao đóng bắc cầu bằng bitset:** tính tập ứng viên của mỗi người như một số nguyên, hợp các tập của người giàu hơn trực tiếp (OR), rồi lấy người yên lặng nhất.
- **Xử lý người yên lặng nhất trước:** nếu `p` là người yên lặng nhất chưa được xét thì **mọi người nghèo hơn hoặc bằng `p`** (đến được từ `p` theo `giàu -> nghèo`) mà chưa có đáp án sẽ nhận `p` làm đáp án; mỗi người được gán đúng một lần.

| Cách | Ý tưởng |
|---|---|
| 2 | DFS có nhớ: `ans[i] = argmin(quiet)` của `i` và `ans[j]` của các `j` giàu hơn trực tiếp |
| 3 | Sắp xếp tô-pô (Kahn) giàu -> nghèo; đẩy `ans[u]` xuống người nghèo hơn kề |
| 4 | Bao đóng bắc cầu bằng bitset (số nguyên) rồi chọn bit có `quiet` nhỏ nhất |
| 5 | Duyệt theo `quiet` tăng dần; từ mỗi `p` lan sang người nghèo hơn chưa có đáp án |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS có nhớ

**Giải thích từng đoạn.**

- `g[b]` = danh sách người giàu hơn `b` trực tiếp.
- `ans = [-1] * n`. `dfs(i)`: nếu `ans[i] != -1` thì thôi (đã tính). Ngược lại đặt `ans[i] = i`; với mỗi `j` trong `g[i]`: `dfs(j)` (để `ans[j]` có giá trị) rồi nếu `quiet[ans[j]] < quiet[ans[i]]` thì `ans[i] = ans[j]`.
- Vì đồ thị là DAG nên không có đệ quy vô hạn; độ sâu tối đa `n = 500`.

In [10]:
def loud_dfs(richer: List[List[int]], quiet: List[int]) -> List[int]:
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    ans = [-1] * n

    def dfs(i: int) -> None:
        if ans[i] != -1:
            return
        ans[i] = i
        for j in g[i]:
            dfs(j)
            if quiet[ans[j]] < quiet[ans[i]]:
                ans[i] = ans[j]

    for i in range(n):
        dfs(i)
    return ans

for richer, quiet, expected in examples:
    assert loud_dfs(richer, quiet) == expected
print("DFS có nhớ OK")

DFS có nhớ OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def loudAndRich(self, richer: List[List[int]], quiet: List[int]) -> List[int]:
        from collections import defaultdict

        def dfs(i: int):
            if ans[i] != -1:
                return
            ans[i] = i
            for j in g[i]:
                dfs(j)
                if quiet[ans[j]] < quiet[ans[i]]:
                    ans[i] = ans[j]

        g = defaultdict(list)
        for a, b in richer:
            g[b].append(a)
        n = len(quiet)
        ans = [-1] * n
        for i in range(n):
            dfs(i)
        return ans

sol_dfs = Solution().loudAndRich
print([sol_dfs(r, q) for r, q, _ in examples])

[[5, 5, 2, 5, 4, 5, 6, 7], [0]]


**Độ phức tạp (Cách 2):** thời gian `O(n + E)` (mỗi đỉnh một lần, mỗi cạnh một lần); bộ nhớ `O(n + E)` cho đồ thị cộng `O(n)` ngăn xếp đệ quy.

---

### 7.2. Cách 3 — Sắp xếp tô-pô (Kahn), đẩy xuống người nghèo hơn

**Ý tưởng khác.** Dùng cạnh `giàu -> nghèo` (`out[a]`) và bậc vào. Thứ tự Kahn đi từ người giàu nhất. Khởi tạo `ans[i] = i`. Khi lấy `u` ra, với mỗi người nghèo hơn kề `v`: nếu `quiet[ans[u]] < quiet[ans[v]]` thì `ans[v] = ans[u]`. Khi `u` được lấy ra, `ans[u]` đã chốt (mọi người giàu hơn `u` đã được xử lý trước).

**Giải thích từng đoạn.**

- `out[a].append(b)`, `indeg[b] += 1`.
- Hàng đợi các đỉnh bậc vào 0 (người giàu nhất mỗi thành phần).
- Mỗi lần lấy `u`: cập nhật `ans[v]` cho `v` trong `out[u]`, rồi giảm `indeg[v]`.

In [12]:
def loud_topo(richer: List[List[int]], quiet: List[int]) -> List[int]:
    n = len(quiet)
    out = defaultdict(list)
    indeg = [0] * n
    for a, b in richer:
        out[a].append(b)
        indeg[b] += 1
    ans = list(range(n))
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in out[u]:
            if quiet[ans[u]] < quiet[ans[v]]:
                ans[v] = ans[u]
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return ans

for richer, quiet, expected in examples:
    assert loud_topo(richer, quiet) == expected
print("sắp xếp tô-pô OK")

sắp xếp tô-pô OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def loudAndRich(self, richer: List[List[int]], quiet: List[int]) -> List[int]:
        from collections import deque
        n = len(quiet)
        out = [[] for _ in range(n)]
        indeg = [0] * n
        for a, b in richer:
            out[a].append(b)
            indeg[b] += 1
        ans = list(range(n))
        q = deque(i for i in range(n) if indeg[i] == 0)
        while q:
            u = q.popleft()
            for v in out[u]:
                if quiet[ans[u]] < quiet[ans[v]]:
                    ans[v] = ans[u]
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return ans

sol_topo = Solution().loudAndRich
print([sol_topo(r, q) for r, q, _ in examples])

[[5, 5, 2, 5, 4, 5, 6, 7], [0]]


**Độ phức tạp (Cách 3):** thời gian `O(n + E)`, bộ nhớ `O(n + E)`; không dùng đệ quy.

---

### 7.3. Cách 4 — Bao đóng bắc cầu bằng bitset

**Ý tưởng khác.** `anc[x]` là **tập** (bitset) những người chắc chắn giàu hơn hoặc bằng `x`. Theo thứ tự giàu -> nghèo (hoặc đệ quy có nhớ): `anc[x] = (1 << x) | OR của anc[y]` với `y` giàu hơn `x` trực tiếp. Rồi `answer[x]` = người có `quiet` nhỏ nhất trong bit của `anc[x]`.

**Giải thích từng đoạn.**

- Dùng thứ tự tô-pô giàu -> nghèo (`topo_rich_to_poor`) để khi tính `anc[x]` mọi `y` giàu hơn đã có.
- Chọn người yên lặng nhất: duyệt bit của `anc[x]`.
- Bộ nhớ `O(n^2 / 64)` từ các số nguyên 500 bit.

In [14]:
def loud_bitset(richer: List[List[int]], quiet: List[int]) -> List[int]:
    n = len(quiet)
    parents = [[] for _ in range(n)]            # parents[b] = người giàu hơn b trực tiếp
    for a, b in richer:
        parents[b].append(a)
    order = topo_rich_to_poor(n, richer)
    anc = [0] * n
    for x in order:
        mask = 1 << x
        for y in parents[x]:
            mask |= anc[y]
        anc[x] = mask
    ans = []
    for x in range(n):
        mask, best = anc[x], x
        while mask:
            low = mask & -mask
            p = low.bit_length() - 1
            if quiet[p] < quiet[best]:
                best = p
            mask ^= low
        ans.append(best)
    return ans

for richer, quiet, expected in examples:
    assert loud_bitset(richer, quiet) == expected
print("bitset OK")

bitset OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def loudAndRich(self, richer: List[List[int]], quiet: List[int]) -> List[int]:
        from collections import deque
        n = len(quiet)
        parents = [[] for _ in range(n)]
        out = [[] for _ in range(n)]
        indeg = [0] * n
        for a, b in richer:
            parents[b].append(a)
            out[a].append(b)
            indeg[b] += 1
        q = deque(i for i in range(n) if indeg[i] == 0)
        anc = [0] * n
        while q:
            x = q.popleft()
            mask = 1 << x
            for y in parents[x]:
                mask |= anc[y]
            anc[x] = mask
            for v in out[x]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        ans = []
        for x in range(n):
            mask, best = anc[x], x
            while mask:
                low = mask & -mask
                p = low.bit_length() - 1
                if quiet[p] < quiet[best]:
                    best = p
                mask ^= low
            ans.append(best)
        return ans

sol_bitset = Solution().loudAndRich
print([sol_bitset(r, q) for r, q, _ in examples])

[[5, 5, 2, 5, 4, 5, 6, 7], [0]]


**Độ phức tạp (Cách 4):** thời gian `O(E * n / 64 + n^2)` (hợp bitset `n / 64` từ máy mỗi cạnh, cộng duyệt bit); bộ nhớ `O(n^2 / 64)`.

---

### 7.4. Cách 5 — Xử lý theo độ yên lặng tăng dần

**Ý tưởng khác.** Với người `x`, đáp án là người yên lặng nhất trong tập ứng viên. Xét người yên lặng nhất **cả nhóm** `p`: mọi người nghèo hơn hoặc bằng `p` đều có `p` trong ứng viên và `p` yên lặng nhất tuyệt đối, nên đáp án của họ là `p`. Loại họ khỏi bài toán và lặp lại với người yên lặng nhất còn lại.

**Giải thích từng đoạn.**

- `out[a]` = người nghèo hơn `a` trực tiếp (cạnh `giàu -> nghèo`).
- Sắp các người theo `quiet` tăng dần. Với mỗi `p`: nếu `ans[p] == -1` thì BFS/DFS từ `p` theo `out`, gán `ans = p` cho mọi đỉnh **chưa có đáp án** gặp được (và không đi tiếp qua đỉnh đã có đáp án: người đó và mọi người nghèo hơn họ đã được gán bởi người yên lặng hơn).
- Mỗi đỉnh được gán đúng một lần.

In [16]:
def loud_by_quiet(richer: List[List[int]], quiet: List[int]) -> List[int]:
    n = len(quiet)
    out = [[] for _ in range(n)]
    for a, b in richer:
        out[a].append(b)
    ans = [-1] * n
    for p in sorted(range(n), key=lambda i: quiet[i]):
        if ans[p] != -1:
            continue
        ans[p] = p
        stack = [p]
        while stack:
            u = stack.pop()
            for v in out[u]:
                if ans[v] == -1:
                    ans[v] = p
                    stack.append(v)
    return ans

for richer, quiet, expected in examples:
    assert loud_by_quiet(richer, quiet) == expected
print("theo độ yên lặng OK")

theo độ yên lặng OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def loudAndRich(self, richer: List[List[int]], quiet: List[int]) -> List[int]:
        n = len(quiet)
        out = [[] for _ in range(n)]
        for a, b in richer:
            out[a].append(b)
        ans = [-1] * n
        for p in sorted(range(n), key=lambda i: quiet[i]):
            if ans[p] != -1:
                continue
            ans[p] = p
            stack = [p]
            while stack:
                u = stack.pop()
                for v in out[u]:
                    if ans[v] == -1:
                        ans[v] = p
                        stack.append(v)
        return ans

sol_quiet = Solution().loudAndRich
print([sol_quiet(r, q) for r, q, _ in examples])

[[5, 5, 2, 5, 4, 5, 6, 7], [0]]


**Độ phức tạp (Cách 5):** thời gian `O(n log n + n + E)` (sắp xếp, rồi mỗi đỉnh và mỗi cạnh một lần); bộ nhớ `O(n + E)`.

## 8. TRACE từng bước

### 8.1. Cách 2 (DFS có nhớ) trên ví dụ 1

`g[b]` = người giàu hơn `b` trực tiếp: `g[0] = [1]`, `g[1] = [2, 3]`, `g[3] = [4, 5, 6]`, `g[7] = [3]`. `quiet = [3, 2, 5, 4, 6, 1, 7, 0]`.

`dfs(0)`: `ans[0] = 0`; `dfs(1)`: `ans[1] = 1`; `dfs(2)`: `ans[2] = 2` (không có ai giàu hơn) -> `quiet[2] = 5 < quiet[1] = 2`? không. `dfs(3)`: `ans[3] = 3`, `dfs(4)`, `dfs(5)`, `dfs(6)` đều là gốc; `quiet[5] = 1 < quiet[3] = 4` -> `ans[3] = 5`. Quay lại `1`: `quiet[ans[3]] = quiet[5] = 1 < quiet[1] = 2` -> `ans[1] = 5`. Quay lại `0`: `quiet[5] = 1 < quiet[0] = 3` -> `ans[0] = 5`.

In [18]:
def trace_dfs(richer, quiet):
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    ans = [-1] * n

    def dfs(i, depth):
        pad = "  " * depth
        if ans[i] != -1:
            print(f"{pad}dfs({i}): đã có ans[{i}] = {ans[i]} -> bỏ qua")
            return
        ans[i] = i
        print(f"{pad}dfs({i}): ans[{i}] = {i} (quiet={quiet[i]}); giàu hơn trực tiếp: {g[i]}")
        for j in g[i]:
            dfs(j, depth + 1)
            if quiet[ans[j]] < quiet[ans[i]]:
                print(f"{pad}  quiet[ans[{j}]={ans[j]}]={quiet[ans[j]]} < quiet[ans[{i}]={ans[i]}]={quiet[ans[i]]} -> ans[{i}] = {ans[j]}")
                ans[i] = ans[j]
    for i in range(n):
        dfs(i, 0)
    print("kết quả:", ans)
    return ans

assert trace_dfs(*examples[0][:2]) == examples[0][2]

dfs(0): ans[0] = 0 (quiet=3); giàu hơn trực tiếp: [1]
  dfs(1): ans[1] = 1 (quiet=2); giàu hơn trực tiếp: [2, 3]
    dfs(2): ans[2] = 2 (quiet=5); giàu hơn trực tiếp: []
    dfs(3): ans[3] = 3 (quiet=4); giàu hơn trực tiếp: [4, 5, 6]
      dfs(4): ans[4] = 4 (quiet=6); giàu hơn trực tiếp: []
      dfs(5): ans[5] = 5 (quiet=1); giàu hơn trực tiếp: []
      quiet[ans[5]=5]=1 < quiet[ans[3]=3]=4 -> ans[3] = 5
      dfs(6): ans[6] = 6 (quiet=7); giàu hơn trực tiếp: []
    quiet[ans[3]=5]=1 < quiet[ans[1]=1]=2 -> ans[1] = 5
  quiet[ans[1]=5]=1 < quiet[ans[0]=0]=3 -> ans[0] = 5
dfs(1): đã có ans[1] = 5 -> bỏ qua
dfs(2): đã có ans[2] = 2 -> bỏ qua
dfs(3): đã có ans[3] = 5 -> bỏ qua
dfs(4): đã có ans[4] = 4 -> bỏ qua
dfs(5): đã có ans[5] = 5 -> bỏ qua
dfs(6): đã có ans[6] = 6 -> bỏ qua
dfs(7): ans[7] = 7 (quiet=0); giàu hơn trực tiếp: [3]
  dfs(3): đã có ans[3] = 5 -> bỏ qua
kết quả: [5, 5, 2, 5, 4, 5, 6, 7]


### 8.2. Cách 3 (Kahn) trên ví dụ 1

Bậc vào của mỗi người là số người giàu hơn trực tiếp. Hàm dưới đây in bậc vào, thứ tự xử lý và mỗi lần cập nhật `ans`.

In [19]:
def trace_topo(richer, quiet):
    n = len(quiet)
    out = defaultdict(list)
    indeg = [0] * n
    for a, b in richer:
        out[a].append(b)
        indeg[b] += 1
    print("bậc vào:", indeg)
    ans = list(range(n))
    q = deque(i for i in range(n) if indeg[i] == 0)
    print("hàng đợi ban đầu (người giàu nhất mỗi thành phần):", list(q))
    while q:
        u = q.popleft()
        print(f"lấy {u} (ans={ans[u]}, quiet={quiet[ans[u]]}); nghèo hơn trực tiếp: {out[u]}")
        for v in out[u]:
            if quiet[ans[u]] < quiet[ans[v]]:
                print(f"   cập nhật ans[{v}]: {ans[v]} -> {ans[u]}")
                ans[v] = ans[u]
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    print("kết quả:", ans)
    return ans

assert trace_topo(*examples[0][:2]) == examples[0][2]

bậc vào: [1, 2, 0, 3, 0, 0, 0, 1]
hàng đợi ban đầu (người giàu nhất mỗi thành phần): [2, 4, 5, 6]
lấy 2 (ans=2, quiet=5); nghèo hơn trực tiếp: [1]
lấy 4 (ans=4, quiet=6); nghèo hơn trực tiếp: [3]
lấy 5 (ans=5, quiet=1); nghèo hơn trực tiếp: [3]
   cập nhật ans[3]: 3 -> 5
lấy 6 (ans=6, quiet=7); nghèo hơn trực tiếp: [3]
lấy 3 (ans=5, quiet=1); nghèo hơn trực tiếp: [1, 7]
   cập nhật ans[1]: 1 -> 5
lấy 1 (ans=5, quiet=1); nghèo hơn trực tiếp: [0]
   cập nhật ans[0]: 0 -> 5
lấy 7 (ans=7, quiet=0); nghèo hơn trực tiếp: []
lấy 0 (ans=5, quiet=1); nghèo hơn trực tiếp: []
kết quả: [5, 5, 2, 5, 4, 5, 6, 7]


### 8.3. Cách 4: bitset ứng viên của mỗi người

Hàm in `anc[x]` dưới dạng tập chỉ số cho mỗi người:

In [20]:
def trace_bitset(richer, quiet):
    n = len(quiet)
    parents = [[] for _ in range(n)]
    for a, b in richer:
        parents[b].append(a)
    anc = [0] * n
    for x in topo_rich_to_poor(n, richer):
        mask = 1 << x
        for y in parents[x]:
            mask |= anc[y]
        anc[x] = mask
    for x in range(n):
        members = [i for i in range(n) if anc[x] >> i & 1]
        best = min(members, key=lambda p: quiet[p])
        print(f"x={x}: ứng viên {members} -> {best} (quiet {quiet[best]})")
    return [min([i for i in range(n) if anc[x] >> i & 1], key=lambda p: quiet[p]) for x in range(n)]

assert trace_bitset(*examples[0][:2]) == examples[0][2]

x=0: ứng viên [0, 1, 2, 3, 4, 5, 6] -> 5 (quiet 1)
x=1: ứng viên [1, 2, 3, 4, 5, 6] -> 5 (quiet 1)
x=2: ứng viên [2] -> 2 (quiet 5)
x=3: ứng viên [3, 4, 5, 6] -> 5 (quiet 1)
x=4: ứng viên [4] -> 4 (quiet 6)
x=5: ứng viên [5] -> 5 (quiet 1)
x=6: ứng viên [6] -> 6 (quiet 7)
x=7: ứng viên [3, 4, 5, 6, 7] -> 7 (quiet 0)


### 8.4. Cách 5: người yên lặng nhất lan xuống

`quiet = [3, 2, 5, 4, 6, 1, 7, 0]` -> thứ tự theo quiet tăng dần: `7, 5, 1, 0, 3, 2, ...`. Xử lý `7` (quiet 0): lan tới người nghèo hơn 7 theo `out`: không ai (7 không là `a` của cạnh nào). `ans[7] = 7`. Xử lý `5`: lan tới `3`, `1`, `0` (và `7`? `3 -> 7` nên 7 đã có `ans`). Cứ thế.

In [21]:
def trace_by_quiet(richer, quiet):
    n = len(quiet)
    out = [[] for _ in range(n)]
    for a, b in richer:
        out[a].append(b)
    ans = [-1] * n
    for p in sorted(range(n), key=lambda i: quiet[i]):
        if ans[p] != -1:
            print(f"p={p} (quiet {quiet[p]}): đã có đáp án {ans[p]} -> bỏ qua")
            continue
        ans[p] = p
        reached = [p]
        stack = [p]
        while stack:
            u = stack.pop()
            for v in out[u]:
                if ans[v] == -1:
                    ans[v] = p
                    reached.append(v)
                    stack.append(v)
        print(f"p={p} (quiet {quiet[p]}): gán đáp án p cho {sorted(reached)}")
    print("kết quả:", ans)
    return ans

assert trace_by_quiet(*examples[0][:2]) == examples[0][2]

p=7 (quiet 0): gán đáp án p cho [7]
p=5 (quiet 1): gán đáp án p cho [0, 1, 3, 5]
p=1 (quiet 2): đã có đáp án 5 -> bỏ qua
p=0 (quiet 3): đã có đáp án 5 -> bỏ qua
p=3 (quiet 4): đã có đáp án 5 -> bỏ qua
p=2 (quiet 5): gán đáp án p cho [2]
p=4 (quiet 6): gán đáp án p cho [4]
p=6 (quiet 7): gán đáp án p cho [6]
kết quả: [5, 5, 2, 5, 4, 5, 6, 7]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (DFS có nhớ): ngắn, lời giải chuẩn, đủ nhanh với `n <= 500`. Cách 3 và 5 là hai phương án không đệ quy cùng `O(n + E)`.

```python
class Solution:
    def loudAndRich(self, richer: List[List[int]], quiet: List[int]) -> List[int]:
        def dfs(i: int):
            if ans[i] != -1:
                return
            ans[i] = i
            for j in g[i]:
                dfs(j)
                if quiet[ans[j]] < quiet[ans[i]]:
                    ans[i] = ans[j]

        g = defaultdict(list)
        for a, b in richer:
            g[b].append(a)
        n = len(quiet)
        ans = [-1] * n
        for i in range(n):
            dfs(i)
        return ans
```

In [22]:
class Solution:
    def loudAndRich(self, richer: List[List[int]], quiet: List[int]) -> List[int]:
        def dfs(i: int):
            if ans[i] != -1:
                return
            ans[i] = i
            for j in g[i]:
                dfs(j)
                if quiet[ans[j]] < quiet[ans[i]]:
                    ans[i] = ans[j]

        g = defaultdict(list)
        for a, b in richer:
            g[b].append(a)
        n = len(quiet)
        ans = [-1] * n
        for i in range(n):
            dfs(i)
        return ans

sol = Solution()
for richer, quiet, expected in examples:
    got = sol.loudAndRich(richer, quiet)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

[5, 5, 2, 5, 4, 5, 6, 7] OK
[0] OK


## 10. Vì sao đúng

**(a) Tập ứng viên.** Gọi `C(x)` là tập những người chắc chắn giàu bằng hoặc hơn `x`. Người `y` **chắc chắn** giàu hơn `x` khi và chỉ khi có đường đi `giàu -> nghèo` từ `y` tới `x` trong đồ thị `richer` (vì chỉ các cạnh được cho là chắc chắn; mọi dữ liệu khác không thể suy ra từ chúng). Do đó `C(x) = {x} ∪ ⋃_{y: [y, x] ∈ richer} C(y)`: ứng viên của `x` là chính `x` cộng ứng viên của **mọi người giàu hơn trực tiếp**.

**(b) Công thức đệ quy.** Gọi `ans(x) = argmin_{p ∈ C(x)} quiet[p]`. Với `quiet` đôi một khác nhau, argmin của hợp bằng argmin của các argmin: `ans(x) = argmin(quiet)` trên `{x} ∪ {ans(y) : y giàu hơn x trực tiếp}`. Cách 2 tính đúng công thức này; DFS có nhớ chỉ chạy mỗi đỉnh một lần vì `ans[i] != -1` chặn. Vì DAG, `dfs(j)` không bao giờ chạm lại đỉnh đang xử lý.

**(c) Cách 3.** Theo thứ tự tô-pô (giàu -> nghèo), khi `u` được lấy ra mọi người giàu hơn `u` trực tiếp đã được lấy ra trước (bậc vào của `u` là 0), nên `ans[u]` đã cộng đủ mọi đóng góp của người giàu hơn và đã chốt. Ta đẩy `ans[u]` xuống `v`: `ans[v]` là min trên `{v} ∪ {ans(u): u giàu hơn v trực tiếp}`: đúng công thức (b).

**(d) Cách 4.** `anc[x]` đúng là `C(x)` (hợp bitset theo (a)); chọn argmin `quiet` trên đó theo định nghĩa.

**(e) Cách 5.** Quy nạp theo thứ tự `quiet` tăng dần. Khi xét `p` (chưa có đáp án), mọi người **được gán trước đó** có đáp án là người yên lặng hơn `p`. `p` không thuộc `C(q)` của những người đã được gán trước? Có thể thuộc: nhưng khi đó người gán trước (yên lặng hơn `p`) cũng thuộc `C(q)` và là argmin tốt hơn, đúng. Với người `v` chưa gán và `v` đến được từ `p`: `p ∈ C(v)`; mọi người trong `C(v)` yên lặng hơn `p` đã được xử lý trước, và nếu có một người `r ∈ C(v)` yên lặng hơn `p` thì `r` đã lan tới `v` (hoặc đỉnh trên đường từ `r` tới `v` đã được gán) làm `v` có đáp án; vì `v` chưa gán nên `p` là người yên lặng nhất trong `C(v)`. Dừng lan tại đỉnh đã gán đúng vì mọi người phía nghèo hơn đỉnh đó đã được gán bởi người yên lặng hơn (đỉnh đó nằm trong `C` của họ).

**Kiểm chứng bằng máy:** so mọi cách với oracle (Cách 1, định nghĩa trực tiếp) trên **mọi** DAG nhất quán của 4 người (mọi tập cạnh, mọi hoán vị `quiet`) và trên nhiều DAG ngẫu nhiên.

In [23]:
solutions = {
    "brute": loud_brute,
    "dfs": loud_dfs,
    "topo": loud_topo,
    "bitset": loud_bitset,
    "by_quiet": loud_by_quiet,
    "sol_dfs": sol_dfs,
    "sol_topo": sol_topo,
    "sol_bitset": sol_bitset,
    "sol_quiet": sol_quiet,
    "final": Solution().loudAndRich,
}

from itertools import permutations
checked = 0
n = 4
for wealth_order in permutations(range(n)):                   # wealth_order[0] giàu nhất
    pairs = [(wealth_order[i], wealth_order[j]) for i in range(n) for j in range(i + 1, n)]
    for mask in range(1 << len(pairs)):
        richer = [list(pairs[t]) for t in range(len(pairs)) if mask >> t & 1]
        if wealth_order != tuple(range(n)) and mask % 7 != 0:
            continue                                           # giảm số ca cho thứ tự khác (kiểm bao quát vẫn đủ lớn)
        for quiet in permutations(range(n)):
            quiet = list(quiet)
            ref = loud_brute(richer, quiet)
            for name, fn in solutions.items():
                assert fn(richer, quiet) == ref, (name, richer, quiet, fn(richer, quiet), ref)
            checked += 1
print(f"DAG 4 người (mọi tập cạnh với thứ tự giàu chuẩn, mẫu cho các thứ tự khác) x mọi quiet: {checked} ca x {len(solutions)} cài đặt khớp oracle")

DAG 4 người (mọi tập cạnh với thứ tự giàu chuẩn, mẫu cho các thứ tự khác) x mọi quiet: 7056 ca x 10 cài đặt khớp oracle


## 11. Time / Space Complexity

Gọi `n` là số người (`<= 500`), `E` là số cạnh (`<= n(n-1)/2`).

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt riêng từng người | `O(n (n + E))` | `O(n + E)` |
| 2. DFS có nhớ | `O(n + E)` | `O(n + E)` |
| 3. Sắp xếp tô-pô | `O(n + E)` | `O(n + E)` |
| 4. Bitset | `O(E n / 64 + n^2)` | `O(n^2 / 64)` |
| 5. Theo độ yên lặng | `O(n log n + E)` | `O(n + E)` |

Đo với `n = 500` và nhiều mức cạnh:

In [24]:
rng = random.Random(851)
cases = [random_dag(rng, 500, p) for p in (0.002, 0.01, 0.05, 0.3)]
# chuỗi dài: 0 giàu hơn 1 giàu hơn ... (đệ quy sâu 500, DAG đường thẳng) và đồ thị đầy đủ
chain = [[i, i + 1] for i in range(499)]
cases.append((chain, list(range(500))[::-1]))
full_order = list(range(500))
cases.append(([[i, j] for i in range(500) for j in range(i + 1, 500)], rng.sample(range(500), 500)))

ref = None
for name in ["dfs", "topo", "bitset", "by_quiet"]:
    t0 = time.perf_counter()
    res = [solutions[name](r, q) for r, q in cases]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:9} {len(cases)} đồ thị n=500: {dt:8.1f} ms")
t0 = time.perf_counter()
bres = [loud_brute(r, q) for r, q in cases[:3]]
print(f"brute 3 đồ thị đầu (thưa): {1000 * (time.perf_counter() - t0):8.1f} ms | khớp:", bres == ref[:3])

dfs       6 đồ thị n=500:     31.9 ms
topo      6 đồ thị n=500:     32.5 ms


bitset    6 đồ thị n=500:    141.2 ms
by_quiet  6 đồ thị n=500:     12.4 ms


brute 3 đồ thị đầu (thưa):     57.1 ms | khớp: True


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle, và test ngẫu nhiên có seed (DAG nhất quán, đúng ràng buộc: `quiet` là hoán vị của `0..n-1`, các cặp `richer` khác nhau).

In [25]:
# 12.1 ví dụ đề
for richer, quiet, expected in examples:
    for name, fn in solutions.items():
        assert fn(richer, quiet) == expected, (name, richer, quiet)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([], [0], [0]),                                         # một người
    ([], [1, 0], [0, 1]),                                   # hai người không quan hệ: mỗi người tự chọn mình
    ([[0, 1]], [1, 0], [0, 1]),                             # 0 giàu hơn 1; quiet[1] = 0 < quiet[0] nhưng 1 nghèo hơn
    ([[1, 0]], [1, 0], [1, 1]),                             # 1 giàu hơn 0; 1 yên lặng hơn: answer[0] = 1
    ([[0, 1], [1, 2]], [0, 1, 2], [0, 0, 0]),               # chuỗi 0 > 1 > 2, 0 yên lặng nhất
    ([[0, 1], [1, 2]], [2, 1, 0], [0, 1, 2]),               # quiet giảm dần theo độ nghèo: mỗi người yên lặng nhất trong tập của mình là chính họ
    ([[0, 2], [1, 2]], [2, 0, 1], [0, 1, 1]),               # 2 có hai người giàu hơn: 1 (quiet 0) yên lặng hơn 0 (quiet 2)
    ([[0, 1], [0, 2], [0, 3]], [0, 3, 2, 1], [0, 0, 0, 0]), # hình sao: 0 giàu nhất và yên lặng nhất
    ([[3, 0], [3, 1], [3, 2]], [0, 1, 2, 3], [0, 1, 2, 3]), # 3 giàu nhất nhưng ồn nhất (quiet lớn nhất)
    ([[0, 1], [2, 1]], [1, 2, 0], [0, 2, 2]),               # 1 có hai người giàu hơn; 2 yên lặng hơn 0
]
for richer, quiet, expected in edge:
    assert loud_brute(richer, quiet) == expected, ("oracle khác kỳ vọng", richer, quiet, loud_brute(richer, quiet))
    for name, fn in solutions.items():
        assert fn(richer, quiet) == expected, (name, richer, quiet, fn(richer, quiet), expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 10 ca


In [26]:
# 12.3 phản ví dụ: các cách sai
def wrong_reversed_edges(richer, quiet):       # đặt ngược chiều cạnh: g[a].append(b)
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[a].append(b)
    ans = [-1] * n
    def dfs(i):
        if ans[i] != -1:
            return
        ans[i] = i
        for j in g[i]:
            dfs(j)
            if quiet[ans[j]] < quiet[ans[i]]:
                ans[i] = ans[j]
    for i in range(n):
        dfs(i)
    return ans

def wrong_no_self(richer, quiet):              # quên chính x trong tập ứng viên
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    ans = []
    for x in range(n):
        seen, stack = set(), [x]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
        ans.append(min(seen, key=lambda p: quiet[p]) if seen else -1)
    return ans

def wrong_direct_only(richer, quiet):          # chỉ xét người giàu hơn trực tiếp (không bắc cầu)
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    return [min([x] + g[x], key=lambda p: quiet[p]) for x in range(n)]

def wrong_compare_ids(richer, quiet):          # so sánh chỉ số thay vì quiet
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    ans = [-1] * n
    def dfs(i):
        if ans[i] != -1:
            return
        ans[i] = i
        for j in g[i]:
            dfs(j)
            if ans[j] < ans[i]:                # SAI: so chỉ số
                ans[i] = ans[j]
    for i in range(n):
        dfs(i)
    return ans

wrongs = [
    ("đảo ngược chiều cạnh", wrong_reversed_edges, [[1, 0], [2, 1]], [2, 1, 0]),
    ("quên chính x", wrong_no_self, [], [0]),
    ("không bắc cầu", wrong_direct_only, [[1, 0], [2, 1]], [2, 1, 0]),
    ("so sánh chỉ số thay vì quiet", wrong_compare_ids, [[1, 0]], [1, 0]),
]
for name, fn, richer, quiet in wrongs:
    truth = loud_brute(richer, quiet)
    print(f"{name:30} richer={richer} quiet={quiet}: sai = {fn(richer, quiet)}, đúng = {truth}")
    assert fn(richer, quiet) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

đảo ngược chiều cạnh           richer=[[1, 0], [2, 1]] quiet=[2, 1, 0]: sai = [0, 1, 2], đúng = [2, 2, 2]
quên chính x                   richer=[] quiet=[0]: sai = [-1], đúng = [0]
không bắc cầu                  richer=[[1, 0], [2, 1]] quiet=[2, 1, 0]: sai = [1, 2, 2], đúng = [2, 2, 2]
so sánh chỉ số thay vì quiet   richer=[[1, 0]] quiet=[1, 0]: sai = [0, 1], đúng = [1, 1]
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [27]:
# 12.4 phản ví dụ về hiệu năng: DFS KHÔNG nhớ chạy mũ trên DAG dạng "tầng đầy đủ nối tiếp"
def count_calls_no_memo(richer, quiet):
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    calls = 0
    def dfs(i):
        nonlocal calls
        calls += 1
        best = i
        for j in g[i]:
            r = dfs(j)
            if quiet[r] < quiet[best]:
                best = r
        return best
    for i in range(n):
        dfs(i)
    return calls

def layered_dag(layers, width=2):
    richer = []
    for L in range(layers - 1):
        for a in range(width):
            for b in range(width):
                richer.append([L * width + a, (L + 1) * width + b])
    return richer, list(range(layers * width))

for layers in (4, 8, 12, 16):
    r, q = layered_dag(layers)
    print(f"{layers:2} tầng x 2: số lần gọi dfs không nhớ = {count_calls_no_memo(r, q):>10,} | số đỉnh = {len(q)}")

 4 tầng x 2: số lần gọi dfs không nhớ =         52 | số đỉnh = 8
 8 tầng x 2: số lần gọi dfs không nhớ =      1,004 | số đỉnh = 16
12 tầng x 2: số lần gọi dfs không nhớ =     16,356 | số đỉnh = 24
16 tầng x 2: số lần gọi dfs không nhớ =    262,108 | số đỉnh = 32


Khi không ghi nhớ, số lần gọi tăng **gấp đôi** mỗi tầng (mỗi đỉnh có hai cha, mỗi cha lại có hai cha, ...): nổ mũ. Với `n = 500` và `E` lớn thì không thể. **Đúng:** `if ans[i] != -1: return`.

In [28]:
# 12.5 ngẫu nhiên nhỏ so với oracle
rng = random.Random(2851)
for _ in range(400):
    n = rng.randint(1, 12)
    richer, quiet = random_dag(rng, n, rng.choice([0.0, 0.1, 0.3, 0.7, 1.0]))
    rng.shuffle(richer)
    ref = loud_brute(richer, quiet)
    for name, fn in solutions.items():
        assert fn(richer, quiet) == ref, (name, richer, quiet)
print("12.5 ngẫu nhiên nhỏ so với oracle: OK")

# 12.6 ngẫu nhiên lớn (n tới 500): các cách khớp nhau; kiểm tra thêm bằng tính chất của đáp án
rng = random.Random(77)
for _ in range(25):
    n = rng.randint(50, 500)
    p = rng.choice([0.002, 0.01, 0.05, 0.2])
    richer, quiet = random_dag(rng, n, p)
    rng.shuffle(richer)
    res = {name: fn(richer, quiet) for name, fn in solutions.items() if name != "brute"}
    assert all(v == res["final"] for v in res.values()), n
    ans = res["final"]
    # tính chất: quiet[ans[x]] <= quiet[x]; và nếu a giàu hơn b trực tiếp thì quiet[ans[b]] <= quiet[ans[a]]
    assert all(quiet[ans[x]] <= quiet[x] for x in range(n))
    assert all(quiet[ans[b]] <= quiet[ans[a]] for a, b in richer)
print("12.6 ngẫu nhiên lớn và tính chất đơn điệu: OK")

# 12.7 đối chiếu brute trên vài đồ thị n tới 120
rng = random.Random(8)
for _ in range(10):
    n = rng.randint(60, 120)
    richer, quiet = random_dag(rng, n, rng.choice([0.01, 0.05, 0.2]))
    assert loud_brute(richer, quiet) == Solution().loudAndRich(richer, quiet)
print("12.7 đối chiếu oracle với n tới 120: OK")

12.5 ngẫu nhiên nhỏ so với oracle: OK


12.6 ngẫu nhiên lớn và tính chất đơn điệu: OK
12.7 đối chiếu oracle với n tới 120: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đặt ngược chiều cạnh

Ta cần đi từ `x` tới **người giàu hơn**. Nếu lưu `g[a].append(b)` (giàu -> nghèo) thì DFS đi tới người **nghèo** hơn.

In [29]:
richer, quiet = [[1, 0], [2, 1]], [2, 1, 0]
print("sai (ngược cạnh):", wrong_reversed_edges(richer, quiet), "| đúng:", Solution().loudAndRich(richer, quiet))
assert wrong_reversed_edges(richer, quiet) != Solution().loudAndRich(richer, quiet)

sai (ngược cạnh): [0, 1, 2] | đúng: [2, 2, 2]


Vì sao sai: `answer[x]` chỉ được chọn trong những người giàu bằng hoặc hơn `x`; cạnh đảo chiều cho tập những người nghèo hơn. **Đúng:** `g[b].append(a)` (với `[a, b]` nghĩa `a` giàu hơn `b`).

### Lỗi 2 — Quên chính `x`

Người `x` cũng thuộc tập "giàu bằng hoặc hơn `x`". Nếu không ai giàu hơn `x`, đáp án phải là `x`.

In [30]:
print("sai (không có x):", wrong_no_self([], [0]), "| đúng:", Solution().loudAndRich([], [0]))
assert wrong_no_self([], [0]) == [-1] and Solution().loudAndRich([], [0]) == [0]

sai (không có x): [-1] | đúng: [0]


Vì sao sai: tập ứng viên rỗng. **Đúng:** khởi tạo `ans[i] = i` (hoặc bắt đầu tập bằng `{x}`).

### Lỗi 3 — Chỉ xét người giàu hơn trực tiếp

"Chắc chắn giàu hơn" bao gồm **gián tiếp qua chuỗi** `a > b > c`.

In [31]:
richer, quiet = [[1, 0], [2, 1]], [2, 1, 0]
print("sai (trực tiếp):", wrong_direct_only(richer, quiet), "| đúng:", Solution().loudAndRich(richer, quiet))
assert wrong_direct_only(richer, quiet) == [1, 2, 2] and Solution().loudAndRich(richer, quiet) == [2, 2, 2]

sai (trực tiếp): [1, 2, 2] | đúng: [2, 2, 2]


Vì sao sai: với `2 > 1 > 0`, người 2 chắc chắn giàu hơn 0 nhưng cặp `(2, 0)` không được cho; chỉ xét cặp trực tiếp bỏ sót người 2 cho `answer[0]`. **Đúng:** dùng DFS/DP để lan tỏa kết quả (`ans[j]` đã bao gồm người giàu hơn `j`).

### Lỗi 4 — So sánh chỉ số thay vì `quiet`

Điều kiện cập nhật phải là `quiet[ans[j]] < quiet[ans[i]]`.

In [32]:
richer, quiet = [[1, 0]], [1, 0]
print("sai (so chỉ số):", wrong_compare_ids(richer, quiet), "| đúng:", Solution().loudAndRich(richer, quiet))
assert wrong_compare_ids(richer, quiet) == [0, 1] and Solution().loudAndRich(richer, quiet) == [1, 1]

sai (so chỉ số): [0, 1] | đúng: [1, 1]


Vì sao sai: người 1 giàu hơn người 0 và yên lặng hơn (`quiet[1] = 0 < quiet[0] = 1`) nên `answer[0] = 1`, dù chỉ số `1 > 0`. **Đúng:** so sánh `quiet`.

### Lỗi 5 — DFS không ghi nhớ

Mỗi người có nhiều người giàu hơn trực tiếp, mỗi người đó lại có nhiều người giàu hơn: số đường đi tăng mũ. Đã minh họa ở mục 12.4.

In [33]:
r, q = layered_dag(14)
t0 = time.perf_counter()
calls = count_calls_no_memo(r, q)
t1 = time.perf_counter()
res = Solution().loudAndRich(r, q)
t2 = time.perf_counter()
print(f"không nhớ: {calls:,} lần gọi, {1000 * (t1 - t0):.1f} ms | có nhớ: {len(q)} lần tính, {1000 * (t2 - t1):.2f} ms")
assert calls > 100 * len(q)

không nhớ: 65,504 lần gọi, 8.6 ms | có nhớ: 28 lần tính, 0.10 ms


Vì sao chậm: cùng một đỉnh bị duyệt lại qua mọi đường dẫn khác nhau. **Đúng:** `ans[i] != -1` làm điều kiện dừng ngay đầu hàm.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chắc chắn hơn/giàu hơn" (quan hệ bắc cầu), "mọi người đến được" -> đồ thị có hướng và **đến được (reachability)**.
- **DAG + một giá trị cần "min/max/tổng trên mọi tổ tiên"** -> quy hoạch động theo thứ tự tô-pô hoặc DFS có nhớ.
- **Chọn chiều cạnh theo câu hỏi:** câu hỏi "ai giàu hơn x" -> cạnh từ `x` tới người giàu hơn.
- **Giá trị `quiet` khác nhau** làm argmin duy nhất; nếu có trùng cần quy tắc phá hòa.
- Cách "xử lý theo thứ tự giá trị tăng dần rồi lan" là kỹ thuật chung cho các bài "tô màu theo ưu tiên".
- Bitset bằng số nguyên Python tiện cho `n <= 500..2000` khi cần hợp/giao tập.
- Cùng họ: Course Schedule (207, 210), Find Eventual Safe States (802), Longest Increasing Path in a Matrix (329), All Ancestors of a Node in a DAG (2192).
- Test nên có: không có cạnh, chuỗi dài, hình sao, người giàu nhất ồn nhất, đồ thị đầy đủ.

## 15. Biến thể

1. **Số người chắc chắn giàu hơn mỗi người** (bắc cầu): bài tập 5.
2. **Chuỗi giàu dài nhất** (độ dài đường đi dài nhất trong DAG): DP tô-pô.
3. **All Ancestors of a Node in a DAG (2192):** trả về danh sách tổ tiên: bitset hoặc DFS.
4. **Đáp án mọi người cùng quiet nhỏ nhất** nếu `quiet` có trùng: cần thêm quy tắc.

Dưới đây là code cho biến thể 1 (đếm người giàu hơn) và 2 (chuỗi dài nhất), kiểm bằng oracle.

In [34]:
def count_richer(n: int, richer: List[List[int]]) -> List[int]:
    parents = [[] for _ in range(n)]
    for a, b in richer:
        parents[b].append(a)
    anc = [0] * n
    for x in topo_rich_to_poor(n, richer):
        mask = 1 << x
        for y in parents[x]:
            mask |= anc[y]
        anc[x] = mask
    return [anc[x].bit_count() - 1 for x in range(n)]          # trừ chính x

def longest_wealth_chain(n: int, richer: List[List[int]]) -> int:
    out = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in richer:
        out[a].append(b)
        indeg[b] += 1
    length = [1] * n
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in out[u]:
            length[v] = max(length[v], length[u] + 1)
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return max(length)

def count_richer_oracle(n, richer):
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    res = []
    for x in range(n):
        seen, stack = set(), [x]
        while stack:
            u = stack.pop()
            for v in g[u]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
        res.append(len(seen))
    return res

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 10)
    richer, _q = random_dag(rng, n, rng.choice([0.1, 0.3, 0.7]))
    assert count_richer(n, richer) == count_richer_oracle(n, richer)
print(count_richer(8, examples[0][0]), "| chuỗi giàu dài nhất ví dụ 1:", longest_wealth_chain(8, examples[0][0]))
assert longest_wealth_chain(3, [[0, 1], [1, 2]]) == 3 and longest_wealth_chain(3, []) == 1
print("Biến thể: OK")

[6, 5, 0, 3, 0, 0, 0, 4] | chuỗi giàu dài nhất ví dụ 1: 4
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt riêng từng người | Thu thập tổ tiên bằng DFS cho mỗi `x` | `O(n(n+E))` | `O(n+E)` | Oracle |
| 2. DFS có nhớ | `ans[i]` từ `ans[j]` của người giàu hơn | `O(n+E)` | `O(n+E)` | **Cách nộp chính** |
| 3. Sắp xếp tô-pô | Kahn, đẩy `ans` xuống người nghèo hơn | `O(n+E)` | `O(n+E)` | Tránh đệ quy sâu |
| 4. Bitset | Tập tổ tiên bằng số nguyên | `O(En/64 + n^2)` | `O(n^2/64)` | Khi cần chính tập tổ tiên (đếm, liệt kê) |
| 5. Theo độ yên lặng | Người yên lặng nhất lan xuống | `O(n log n + E)` | `O(n+E)` | Khi giá trị ưu tiên (`quiet`) là trung tâm |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
g = defaultdict(list)
for a, b in [[1, 0], [2, 1], [3, 1]]:
    g[b].append(a)
print(dict(g))
print(richer_than(1))   # dùng g của ví dụ 1 ở mục 3.2
print(Solution().loudAndRich([[1, 0]], [1, 0]))
```

In [35]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `richer = [[0, 1], [1, 2], [3, 2]]`, `quiet = [3, 2, 0, 1]`: ghi tập "chắc chắn giàu bằng hoặc hơn" của người 2 và `answer` cho cả 4 người.

In [36]:
# Bài 2: điền kết quả tính tay
hand_candidates_2 = None   # tập đã sắp xếp, ví dụ [0, 1, ...]
hand_answer = None         # list 4 phần tử

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DFS có nhớ.

In [37]:
def exercise_3_dfs(richer: List[List[int]], quiet: List[int]) -> List[int]:
    n = len(quiet)
    g = defaultdict(list)
    # TODO 1: với mỗi [a, b] trong richer, thêm a vào g[b] (a giàu hơn b)
    ans = [-1] * n

    def dfs(i: int) -> None:
        # TODO 2: nếu ans[i] đã có giá trị thì return
        # TODO 3: đặt ans[i] = i
        # TODO 4: với mỗi j trong g[i]: gọi dfs(j); nếu quiet[ans[j]] < quiet[ans[i]] thì ans[i] = ans[j]
        pass

    for i in range(n):
        dfs(i)
    return ans


# assert exercise_3_dfs([[1, 0], [2, 1], [3, 1], [3, 7], [4, 3], [5, 3], [6, 3]], [3, 2, 5, 4, 6, 1, 7, 0]) == [5, 5, 2, 5, 4, 5, 6, 7]
# assert exercise_3_dfs([], [0]) == [0]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_topo(richer, quiet)` bằng sắp xếp tô-pô (Kahn) với cạnh `giàu -> nghèo`.

In [38]:
def exercise_4_topo(richer: List[List[int]], quiet: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_4_topo([[1, 0], [2, 1], [3, 1], [3, 7], [4, 3], [5, 3], [6, 3]], [3, 2, 5, 4, 6, 1, 7, 0]) == [5, 5, 2, 5, 4, 5, 6, 7]
# assert exercise_4_topo([], [0]) == [0]

### Bài 5 — Mini challenge: đếm số người chắc chắn giàu hơn (khó)

Với mỗi `x`, đếm có bao nhiêu người **chắc chắn có nhiều tiền hơn** `x` (bắc cầu, không tính `x`). Trả về list độ dài `n`.

```python
exercise_5_count_richer(3, [[0, 1], [1, 2]])           -> [0, 1, 2]
exercise_5_count_richer(8, richer_của_ví_dụ_1)         -> [6, 5, 0, 3, 0, 0, 0, 4]
```
Gợi ý: bitset (như Cách 4) hoặc DFS có nhớ trả về tập.

In [39]:
def exercise_5_count_richer(n: int, richer: List[List[int]]) -> List[int]:
    # TODO
    pass


# assert exercise_5_count_richer(3, [[0, 1], [1, 2]]) == [0, 1, 2]
# assert exercise_5_count_richer(8, examples[0][0]) == count_richer_oracle(8, examples[0][0])

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `g[b].append(a)` lưu người giàu hơn `b`.
- Mức 2: `[1, 0]`: `g[0] = [1]`; `[2, 1]`: `g[1] = [2]`; `[3, 1]`: `g[1] = [2, 3]`.
- Mức 3: `{0: [1], 1: [2, 3]}`, `richer_than(1)` (với đồ thị ví dụ 1) là `[2, 3, 4, 5, 6]`, và `[1, 1]` cho dòng 3 (người 1 yên lặng hơn người 0 và giàu hơn).

**Bài 2**
- Mức 1: giàu hơn 2: `1` và `3` (trực tiếp); giàu hơn 1: `0`.
- Mức 2: tập của 2 là `{0, 1, 2, 3}`.
- Mức 3: `quiet = [3, 2, 0, 1]`: `answer[0] = 0` (chỉ có chính nó); `answer[1] = 1` (`{0, 1}`: quiet 3 và 2); `answer[2] = 2` (quiet 0); `answer[3] = 3`.

**Bài 3**
- Mức 1: `for a, b in richer: g[b].append(a)`.
- Mức 2: `if ans[i] != -1: return`; `ans[i] = i`.
- Mức 3: vòng `for j in g[i]`: `dfs(j)`; `if quiet[ans[j]] < quiet[ans[i]]: ans[i] = ans[j]`.

**Bài 4**
- Mức 1: `out[a].append(b)`, `indeg[b] += 1`, `ans = list(range(n))`.
- Mức 2: hàng đợi các đỉnh bậc vào 0; lấy `u`, duyệt `v` trong `out[u]`.
- Mức 3: `if quiet[ans[u]] < quiet[ans[v]]: ans[v] = ans[u]`; giảm bậc vào, đưa vào hàng đợi khi về 0.

**Bài 5**
- Mức 1: tập tổ tiên `anc[x]` như Cách 4.
- Mức 2: đếm = `anc[x].bit_count() - 1`.
- Mức 3: thứ tự tính theo tô-pô giàu -> nghèo (`topo_rich_to_poor`).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [40]:
g1 = defaultdict(list)
for a, b in [[1, 0], [2, 1], [3, 1]]:
    g1[b].append(a)
print(dict(g1))
print(richer_than(1))
print(Solution().loudAndRich([[1, 0]], [1, 0]))

assert dict(g1) == {0: [1], 1: [2, 3]}
assert richer_than(1) == [2, 3, 4, 5, 6]
assert Solution().loudAndRich([[1, 0]], [1, 0]) == [1, 1]

{0: [1], 1: [2, 3]}
[2, 3, 4, 5, 6]
[1, 1]


### Đáp án Bài 2

Tập ứng viên của 2: `{0, 1, 2, 3}` (người giàu hơn 2 trực tiếp: `1` và `3`; giàu hơn 1: `0`). `quiet = [3, 2, 0, 1]`: người yên lặng nhất là 2 (quiet 0). `answer = [0, 1, 2, 3]`? Người 0 ở đỉnh giàu nhất: chỉ có chính nó -> 0. Người 1: `{0, 1}` -> `quiet[0] = 3`, `quiet[1] = 2` -> 1. Người 2: `{0,1,2,3}` -> 2 (quiet 0). Người 3: `{3}` -> 3.

In [41]:
richer2, quiet2 = [[0, 1], [1, 2], [3, 2]], [3, 2, 0, 1]
hand_candidates_2_solution = [0, 1, 2, 3]
hand_answer_solution = [0, 1, 2, 3]
g2 = defaultdict(list)
for a, b in richer2:
    g2[b].append(a)
seen, stack = {2}, [2]
while stack:
    u = stack.pop()
    for v in g2[u]:
        if v not in seen:
            seen.add(v)
            stack.append(v)
assert sorted(seen) == hand_candidates_2_solution
assert Solution().loudAndRich(richer2, quiet2) == hand_answer_solution == loud_brute(richer2, quiet2)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_dfs_solution(richer: List[List[int]], quiet: List[int]) -> List[int]:
    n = len(quiet)
    g = defaultdict(list)
    for a, b in richer:
        g[b].append(a)
    ans = [-1] * n

    def dfs(i: int) -> None:
        if ans[i] != -1:
            return
        ans[i] = i
        for j in g[i]:
            dfs(j)
            if quiet[ans[j]] < quiet[ans[i]]:
                ans[i] = ans[j]

    for i in range(n):
        dfs(i)
    return ans

assert exercise_3_dfs_solution(*examples[0][:2]) == examples[0][2]
assert exercise_3_dfs_solution([], [0]) == [0]
rng = random.Random(3)
for _ in range(100):
    richer, quiet = random_dag(rng, rng.randint(1, 10), rng.choice([0.1, 0.4, 0.8]))
    assert exercise_3_dfs_solution(richer, quiet) == loud_brute(richer, quiet)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_topo_solution(richer: List[List[int]], quiet: List[int]) -> List[int]:
    n = len(quiet)
    out = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in richer:
        out[a].append(b)
        indeg[b] += 1
    ans = list(range(n))
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in out[u]:
            if quiet[ans[u]] < quiet[ans[v]]:
                ans[v] = ans[u]
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return ans

assert exercise_4_topo_solution(*examples[0][:2]) == examples[0][2]
assert exercise_4_topo_solution([], [0]) == [0]
rng = random.Random(4)
for _ in range(100):
    richer, quiet = random_dag(rng, rng.randint(1, 10), rng.choice([0.1, 0.4, 0.8]))
    assert exercise_4_topo_solution(richer, quiet) == loud_brute(richer, quiet)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_count_richer_solution(n: int, richer: List[List[int]]) -> List[int]:
    parents = [[] for _ in range(n)]
    for a, b in richer:
        parents[b].append(a)
    anc = [0] * n
    for x in topo_rich_to_poor(n, richer):
        mask = 1 << x
        for y in parents[x]:
            mask |= anc[y]
        anc[x] = mask
    return [anc[x].bit_count() - 1 for x in range(n)]

assert exercise_5_count_richer_solution(3, [[0, 1], [1, 2]]) == [0, 1, 2]
assert exercise_5_count_richer_solution(8, examples[0][0]) == count_richer_oracle(8, examples[0][0])
print("ví dụ 1:", exercise_5_count_richer_solution(8, examples[0][0]))
rng = random.Random(5)
for _ in range(100):
    n = rng.randint(1, 10)
    richer, _q = random_dag(rng, n, rng.choice([0.1, 0.4, 0.8]))
    assert exercise_5_count_richer_solution(n, richer) == count_richer_oracle(n, richer)
print("Bài 5 OK")

ví dụ 1: [6, 5, 0, 3, 0, 0, 0, 4]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DFS có nhớ)

```python
class Solution:
    def loudAndRich(self, richer: List[List[int]], quiet: List[int]) -> List[int]:
        def dfs(i: int):
            if ans[i] != -1:                       # đã tính
                return
            ans[i] = i                             # bắt đầu: chính mình
            for j in g[i]:                         # j giàu hơn i trực tiếp
                dfs(j)
                if quiet[ans[j]] < quiet[ans[i]]:  # so sánh bằng quiet
                    ans[i] = ans[j]

        g = defaultdict(list)
        for a, b in richer:                        # a giàu hơn b: lưu g[b] chứa a
            g[b].append(a)
        n = len(quiet)
        ans = [-1] * n
        for i in range(n):
            dfs(i)
        return ans
```

Công thức nhớ nhanh: `ans[x] = argmin quiet trong {x} ∪ {ans[y] : y giàu hơn x trực tiếp}`.

### Checklist trước khi submit

- [ ] Cạnh lưu theo chiều `g[b].append(a)` (từ người nghèo tới người giàu hơn)?
- [ ] `ans[i] = i` ban đầu (tính cả chính mình)?
- [ ] Có điều kiện dừng `ans[i] != -1` (ghi nhớ) để không chạy mũ?
- [ ] So sánh bằng `quiet[...]`, không so sánh chỉ số?
- [ ] Vòng ngoài gọi `dfs(i)` cho **mọi** `i` (đồ thị có thể không liên thông)?
- [ ] Đã thử: không cạnh, chuỗi dài 500 (đệ quy sâu), đồ thị đầy đủ?
- [ ] Tên method đúng `loudAndRich(self, richer, quiet)` và trả `List[int]`?